# GPU cumulative AO replay
Download a completed Bouchet demo into `results/bouchet/gpu_demo/JOB_ID` first. This notebook reads saved coefficients, timestamps and object estimates; it performs no optimization or inverse object fits. See [README.md](README.md) for queue and download commands.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while not (ROOT / 'optimization').is_dir():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open this notebook inside the repository.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
import importlib
from utils import plotting as plotting_helpers
from systematic_testing.gpu_demo import plot_demo as demo_helpers
from systematic_testing.gpu_demo import replay as replay_helpers
from systematic_testing.ao_animation import viewer as ao_viewer
importlib.reload(plotting_helpers)
importlib.reload(ao_viewer)
importlib.reload(demo_helpers)
importlib.reload(replay_helpers)
from systematic_testing.gpu_demo.plot_demo import load_run, plot_rms, make_plots, make_replay
candidates = list((ROOT / 'results/bouchet/gpu_demo').glob('*/run.json'))
if not candidates:
    raise FileNotFoundError('Download a completed demo result directory.')
RUN_DIR = ROOT / 'results/bouchet/gpu_demo/29480557'
# Override RUN_DIR here to select another run or a prepared preview.
print('Reading:', RUN_DIR)
import json
metadata = json.loads((RUN_DIR / 'run.json').read_text())
history = None
if metadata['status'] == 'prepared':
    print('First-loop data preview; no AO fits have been run.')
else:
    metadata, scene, history = load_run(RUN_DIR)
    print('Status:', metadata['status'], '| Hardware:', metadata.get('hardware', metadata['hostname']))
    print(f"Core: {metadata['core_seconds']:.3f} s; final applied RMS: {history['applied_residual_rms'][-1]:.6f} waves")
    display(pd.DataFrame(metadata['loops'])[['loop', 'fit_images', 'core_seconds', 'screening_seconds', 'accepted_steps', 'success', 'message']])


The curves separate phase-estimation error from the residual after each clipped SLM correction. Core time includes workspace setup, screening and Newton steps; it excludes simulated acquisitions, diagnostics and replay.

In [ ]:
if history is not None:
    fig, ax = plot_rms(metadata, scene, history, by_time=True)
    plt.show()
    fig, ax = plot_rms(metadata, scene, history, by_time=False)
    plt.show()
    print('Static plots:', make_plots(RUN_DIR))


In [ ]:
# Ground-truth sample, acquired z planes, and every diversity used in this run.
fig = demo_helpers.plot_acquisition_geometry(RUN_DIR)
for extension in ('png', 'pdf'):
    fig.savefig(RUN_DIR / 'figures' / f'sample_and_diversities.{extension}', dpi=300)
plt.show()


Replay uses the same image/wavefront viewer as yesterday: true residual, estimated residual and accumulated SLM phase in polar panels; initial/current noisy simulated images at the saved fixed exposure; and the saved RMS trace. Estimates update at every optimizer step. The physical image and SLM change only at applied AO corrections. No optimization, inverse object fit, or voxel rendering is performed. The standalone HTML has play/pause, stepping, a scrubber and speed controls.

In [ ]:
if metadata['status'] == 'prepared':
    frames, points, title = replay_helpers.load_acquisition_preview(RUN_DIR)
    display(ao_viewer.show_replay(frames, points, title=title,
            output_path=RUN_DIR / 'figures/acquisition_preview.html'))
else:
    STRIDE = 1  # Every saved optimizer step; increase to shorten playback.
    FOCUS_MM = 0.0  # z=0 diagnostic with Poisson noise at the saved exposure.
    REPLAY_PATH = RUN_DIR / 'figures' / 'ao_replay.html'
    display(make_replay(RUN_DIR, stride=STRIDE, focus=FOCUS_MM, output_path=REPLAY_PATH))
    print('Standalone viewer:', REPLAY_PATH)


The viewer has already been saved as figures/ao_replay.html inside the selected result folder. Open that file in a browser for standalone playback; no GIF export or additional rendering is required.